# ProteinStruct-ML — Machine Learning Modeling

## Objectives

- Develop regression models to predict protein structural RMSD.
- Establish baseline performance using DummyRegressor.
- Compare linear and nonlinear machine learning algorithms.
- Apply appropriate preprocessing using scikit-learn Pipelines.
- Evaluate model performance using MAE, RMSE, and R².
- Optimize selected models using cross-validation.
- Save the best-performing model for the Streamlit dashboard.

## Inputs

- `data/processed/casp_cleaned.csv`

## Outputs

- Trained regression models
- Model performance comparisons
- Model evaluation visualizations
- Saved prediction pipeline for deployment

## Machine Learning Task

**Type:** Supervised regression

**Target:** RMSD

**Predictors:** F1–F9

The goal is to investigate whether physicochemical descriptors
can predict structural deviation between protein models and
their reference structures.

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
from pathlib import Path

current_dir = Path.cwd()
print("Current directory:", current_dir)

Current directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML\jupyter_notebooks


We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
if Path.cwd().name == "jupyter_notebooks":
    os.chdir(Path.cwd().parent)

print("Working directory set to:", Path.cwd())

Working directory set to: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


Confirm the new current directory

In [4]:
print("Confirmed project directory:", Path.cwd())

Confirmed project directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


## Section 1 — Data Preparation

In [5]:
# Import libraries
import pandas as pd
import numpy as np

# Load the cleaned CASP dataset
df = pd.read_csv("data/processed/casp_cleaned.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (44019, 10)


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [6]:
# Define predictor variables
feature_columns = [f"F{i}" for i in range(1, 10)]

X = df[feature_columns].copy()

# Define target variable
y = df["RMSD"].copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nPredictor columns:")
print(X.columns.tolist())

print("\nTarget variable:")
print(y.name)

Feature matrix shape: (44019, 9)
Target vector shape: (44019,)

Predictor columns:
['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9']

Target variable:
RMSD


### Train/Test Split

The cleaned dataset is divided into training (approximately 80%)
and testing (approximately 20%) subsets.

GroupShuffleSplit is used to prevent identical combinations of
physicochemical descriptors from appearing in both subsets.

The test set will remain separate during model development and
hyperparameter optimization to provide an independent evaluation
of predictive performance.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# Assign the same group identifier to identical feature combinations
groups = pd.util.hash_pandas_object(X, index=False)

# Create an approximately 80/20 group-aware split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Separate training and testing data
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# Verify that identical feature combinations do not cross the split
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

assert train_groups.isdisjoint(test_groups)

print("\nGroup leakage check: PASSED")

Training features: (35210, 9)
Testing features: (8809, 9)
Training target: (35210,)
Testing target: (8809,)

Group leakage check: PASSED


## Section 2 — Baseline Model: DummyRegressor

A DummyRegressor is used to establish baseline predictive performance.

The model predicts the mean RMSD value from the training dataset
without considering the physicochemical descriptors (F1–F9).

This provides a reference against which more sophisticated
regression algorithms can be evaluated.

Model performance is assessed using:

- MAE (Mean Absolute Error): Average absolute prediction error in Å.
- RMSE (Root Mean Squared Error): Penalizes larger prediction errors more strongly, in Å.
- R² (Coefficient of Determination): Measures improvement relative to predicting the mean of the evaluated target values.

In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Initialize baseline model
dummy_model = DummyRegressor(strategy="mean")

# Train using the training dataset only
dummy_model.fit(X_train, y_train)

# Generate predictions on the training dataset
y_train_pred_dummy = dummy_model.predict(X_train)

# Calculate training performance
dummy_mae = mean_absolute_error(y_train, y_train_pred_dummy)
dummy_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_dummy))
dummy_r2 = r2_score(y_train, y_train_pred_dummy)

print("DummyRegressor — Training Performance")
print(f"Mean predicted RMSD: {dummy_model.constant_[0, 0]:.3f} Å")
print(f"MAE:  {dummy_mae:.3f} Å")
print(f"RMSE: {dummy_rmse:.3f} Å")
print(f"R²:   {dummy_r2:.3f}")

DummyRegressor — Training Performance
Mean predicted RMSD: 7.766 Å
MAE:  5.511 Å
RMSE: 6.137 Å
R²:   0.000


### Baseline Model Results and Interpretation

The DummyRegressor was trained using the mean RMSD strategy.

**Training results:**

- Mean predicted RMSD: 7.766 Å
- MAE: 5.511 Å
- RMSE: 6.137 Å
- R²: 0.000

The model predicts the mean RMSD of the training dataset
for every observation, independently of the physicochemical
descriptors.

The MAE of 5.511 Å demonstrates the relatively large
prediction error obtained when the descriptor information
is ignored.

The R² value of 0.000 is expected because predicting the
training mean does not explain any variance in the target.

These results provide an initial reference for evaluating
whether more sophisticated regression algorithms can
extract predictive information from the descriptors.

Subsequent model comparisons will use group-aware
cross-validation to assess generalization performance.

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
